### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the same way humans do—they’re amazing mimics that have evolved a set of traits that make vocal imitation especially easy for them. Here’s why they can copy human speech and why they often do it:\n\n| Reason | How it works | Why it matters for “talking” |\n|--------|--------------|-----------------------------|\n| **Social nature** | Parrots live in flocks where constant vocal interaction maintains group cohesion. In the wild they use a rich repertoire of calls to identify individuals, signal food, danger, or mood. | A captive parrot treats its human family as its “flock.” Mimicking the sounds it hears (including human words) is a way to communicate and bond with its social partners. |\n| **Vocal anatomy** | They have a highly flexible syrinx (the bird equivalent of a voice box) and a muscular tongue and beak that can shape a wide range of sounds. Their brain region called the “song system” is especially developed for learning and reproducing 

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We have a function get_weather(location). We should call it.', 'tool_calls': [{'id': 'fc_991111b5-de61-455d-8a1a-86c73c2c48d1', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 127, 'total_tokens': 180, 'completion_time': 0.110753173, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.00576322, 'prompt_tokens_details': None, 'queue_time': 0.169982524, 'total_time': 0.116516393}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1df16-432f-76cb-afcd-5ef0d4dd072a-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_991111b5-de61-455d-8a1a-86c73c2c48d1', 'type': 'tool_call

### Tool Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston right now.


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. We have function get_weather. Need to call it.', 'tool_calls': [{'id': 'fc_f6564f76-0d48-4d70-8a3f-df468742d941', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 126, 'total_tokens': 172, 'completion_time': 0.096873399, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.005667246, 'prompt_tokens_details': None, 'queue_time': 0.368948283, 'total_time': 0.102540645}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1df1a-1b69-718d-bb39-83024eb3217f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_f6564f76-0d4